# Notebook to plot transmission networks

Input: 
* Solved sec-coupled PyPSA-Earth network (-> e.g. results/SEC-name/postnetworks/elec_s_4_ec_lcopt_Co2L-3H_3H_2050_0.071_NZ_400export.nc)
* Country shapes GeoJSON (-> resources/RUN-name/shapes/country_shapes.geojson)

Does not look good yet? To do:
* Adapt cap_max to fit data (or uncomment the relevant line for automatic setting)
* Adapt legend_caps to fit your needs

## Preparation

In [ ]:
# Choose if you want to plot brownfield or optimized capacities
optimized = False  # False for brownfield

### Load packages and networks

In [ ]:
import yaml
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import pypsa
import warnings
import matplotlib.pyplot as plt
from shapely.geometry import LineString
import contextily as ctx  # Optional
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from cartopy import crs as ccrs
import cartopy.io.img_tiles as cimgt
import cartopy.feature as cfeat
import cartopy.io.shapereader as shpreader
import matplotlib.lines as mlines
import cartopy.feature as cfeature
from pathlib import Path

### Provide network

In [ ]:
# Insert path to network file (.nc) here
n = pypsa.Network("/Users/marcoschamel/git-marco/pypsa-earth/results/EG-sec-WS/postnetworks/elec_s_12_ec_lcopt_Co2L-3H_3H_2050_0.071_NZ_400export.nc") 
regions_onshore = gpd.read_file("/Users/marcoschamel/git-marco/pypsa-earth/resources/EG-sec-v1/shapes/country_shapes.geojson")
gadm_shapes = gpd.read_file("/Users/marcoschamel/git-marco/pypsa-earth/resources/EG-sec-v1/shapes/gadm_shapes.geojson")

#### Extract power lines and build geometries

In [ ]:
# Extract data from the network

elec_buses = {}
elec_lines = {}
elec_capacity = {}
elec_lines_gdf = {}
elec_buses_gdf = {}
elec_line_widths = {}
powerlines = {}

# Define min and max capacity for normalization
cap_min = 0  # in MW
cap_max = 40000  # in MW

# Extract powerlines
powerlines = n.lines.copy()

# Buses
elec_buses = n.buses[n.buses.carrier == 'AC'].copy()
elec_buses_gdf = gpd.GeoDataFrame(
    elec_buses,
    geometry=gpd.points_from_xy(
        elec_buses.x,
        elec_buses.y
    ),
    crs="EPSG:4326"
)

# Powerlines geometry 
def build_powerline_geometry(row):
    x0, y0 = elec_buses.loc[row.bus0, ["x", "y"]]
    x1, y1 = elec_buses.loc[row.bus1, ["x", "y"]]
    return LineString([(x0, y0), (x1, y1)])

powerlines_temp = powerlines.copy()
powerlines_temp["geometry"] = None  # Initialize geometry column first
powerlines_temp["geometry"] = powerlines_temp.apply(build_powerline_geometry, axis=1)
# Write geometry into the original dict as well
powerlines["geometry"] = powerlines_temp["geometry"]
elec_lines_gdf = gpd.GeoDataFrame(powerlines_temp, geometry="geometry", crs="EPSG:4326")

# Set capacity varible name depending on optimized or brownfield plotting
if optimized:
    cap_var = "s_nom_opt"
else:
    cap_var = "s_nom"

# Normalize powerline widths by capacity
if cap_var in elec_lines_gdf.columns:
    elec_capacity = elec_lines_gdf[cap_var]
    # uncomment the following line if you want to scale my max capacity in the dataset
    cap_max = elec_capacity.max()
    elec_line_widths = np.interp(
        elec_capacity,
        (cap_min, cap_max),
        (1.5, 9.0)
    )
else:
    elec_line_widths = [1.5] * len(elec_lines_gdf)

In [ ]:
(n.lines.s_nom_opt - n.lines.s_nom).sort_values(ascending=False).plot.bar(figsize=(10, 5))

#### Plot powerlines

In [ ]:
# Plot powerlines for each scenario side by side
fig, ax = plt.subplots(figsize=(6,6), subplot_kw={"projection": ccrs.PlateCarree()})

# Plot country borders
regions_onshore.plot(ax=ax, color="lightgrey", edgecolor="black")

# Plot powerlines
for geom, width in zip(elec_lines_gdf.geometry, elec_line_widths):
    ax.add_geometries([geom], crs=ccrs.PlateCarree(),
                        linewidth=width, edgecolor="#206bc7", facecolor='none', zorder=2, linestyle='-', capstyle='round')

# Plot buses
ax.scatter(
    elec_buses_gdf.geometry.x,
    elec_buses_gdf.geometry.y,
    s=5,
    color='black',
        zorder=3,
        transform=ccrs.PlateCarree()
    )

# Renove borders around figure
ax.spines['geo'].set_visible(False)

# Legend
legend_caps = [1000, 10000, 30000, 50000]  # in MW
legend_widths = np.interp(legend_caps, (cap_min, cap_max), (1.5, 9.0))
legend_lines = [
    mlines.Line2D([], [], color='#206bc7', linewidth=lw, label=f'{cap / 1000:.1f} GW')
    for cap, lw in zip(legend_caps, legend_widths)
]

legend = fig.legend(
    handles=legend_lines,
    title="Transmission capacity",
    loc="lower left",         
    title_fontproperties={"weight": "bold"}
)

plt.tight_layout()
plt.show()


In [ ]:
# Plot powerlines for each scenario side by side
fig, ax = plt.subplots(figsize=(6,6), subplot_kw={"projection": ccrs.PlateCarree()})

# Plot country borders
regions_onshore.plot(ax=ax, color="lightgrey", edgecolor="black")

# Plot GADM shapes
gadm_shapes.boundary.plot(ax=ax, color="white", linewidth=0.5)

# Filter buses with carrier == 'AC'
ac_buses = n.buses[n.buses.carrier == 'AC']

# For each bus, get generators connected and their p_nom_opt share
bus_pie_data = {}

# Define color mapping for generator carriers
carrier_colors = {
    "solar": "#FFD700",   # yellow
    "onwind": "#206bc7",  # blue
    "offwind": "#00BFFF", # light blue
    "hydro": "#1E8449",   # green
    "OCGT": "#FF5733",    # orange/red
    "CCGT": "#C70039",    # dark red
    # Add more carriers as needed
}

for bus_name, bus in ac_buses.iterrows():
    # Get generators at this bus
    gens = n.generators[n.generators.bus == bus_name]
    # Exclude "load" carrier
    gens = gens[gens.carrier != "load"]
    if len(gens) == 0:
        continue  # Skip if no generators

    # Calculate shares
    total_p_nom_opt = gens.p_nom_opt.sum()
    if total_p_nom_opt == 0:
        continue  # Skip if total is zero

    # Group by carrier and sum p_nom_opt
    carrier_group = gens.groupby("carrier")["p_nom_opt"].sum()
    shares = carrier_group / total_p_nom_opt
    labels = carrier_group.index
    colors = [carrier_colors.get(carrier, "#cccccc") for carrier in labels]

    # Save values for this bus
    bus_pie_data[bus_name] = {
        "shares": shares.values,
        "labels": labels.values,
        "colors": colors,
        "total_p_nom_opt": total_p_nom_opt
    }

    # Plot pie chart at bus location
    x, y = bus['x'], bus['y']
    ax_inset = plt.axes([0,0,0.1,0.1], frameon=False)
    pie = ax_inset.pie(shares, labels=None, colors=colors, startangle=90)
    plt.setp(ax_inset, xticks=[], yticks=[])
    # Move the inset axes to the bus location in data coordinates
    trans = ax.transData.transform((x, y))
    inv = fig.transFigure.inverted()
    x_fig, y_fig = inv.transform(trans)
    ax_inset.set_position([x_fig-0.025, y_fig-0.025, 0.05, 0.05])

plt.show()

In [ ]:
# Calculate total produced energy for each generator (sum over time, weighted)
gen_energy = (n.generators_t.p * n.snapshot_weightings.generators.mean()).sum(axis=0)

# Prepare color mapping (add load carrier from generators)
carrier_colors_with_load = carrier_colors.copy()
carrier_colors_with_load["load"] = "#888888"  # gray for load

# Plot 
fig, ax = plt.subplots(figsize=(6,6), subplot_kw={"projection": ccrs.PlateCarree()})

# Plot country borders
# regions_onshore.plot(ax=ax, color="lightgrey", edgecolor="black")

# Plot GADM shapes
gadm_shapes.plot(ax=ax, color="lightgrey", linewidth=0.5, edgecolor="black")

# For each bus, collect energy by carrier (including generator with carrier == "load")
for bus_name, bus in ac_buses.iterrows():
    # Get generators at this bus
    gens = n.generators[n.generators.bus == bus_name]
    # Group by carrier and sum energy
    gen_energy_at_bus = gen_energy[gens.index].groupby(gens.carrier).sum() if not gens.empty else pd.Series(dtype=float)

    # Only plot if there is any energy
    total_energy = gen_energy_at_bus.sum()
    if total_energy == 0:
        continue

    shares = gen_energy_at_bus / total_energy
    labels = gen_energy_at_bus.index
    colors = [carrier_colors_with_load.get(carrier, "#cccccc") for carrier in labels]

    # Plot pie chart at bus location
    x, y = bus['x'], bus['y']
    ax_inset = plt.axes([0,0,0.1,0.1], frameon=False)
    pie = ax_inset.pie(shares, labels=None, colors=colors, startangle=90)
    plt.setp(ax_inset, xticks=[], yticks=[])
    # Move the inset axes to the bus location in data coordinates
    trans = ax.transData.transform((x, y))
    inv = fig.transFigure.inverted()
    x_fig, y_fig = inv.transform(trans)
    ax_inset.set_position([x_fig-0.025, y_fig-0.025, 0.05, 0.05])

plt.show()


In [ ]:
# Get all generator carriers (types)
all_carriers = n.generators.carrier.unique()

# Get all AC buses
ac_buses = n.buses[n.buses.carrier == 'AC']

# Prepare DataFrame: rows=buses, columns=generator types
gen_energy = (n.generators_t.p * n.snapshot_weightings.generators.mean()).sum(axis=0)
bus_gen_energy = pd.DataFrame(0.0, index=ac_buses.index, columns=all_carriers)

for bus_name in ac_buses.index:
    gens = n.generators[n.generators.bus == bus_name]
    if not gens.empty:
        # Sum energy by carrier at this bus
        gen_energy_at_bus = gen_energy[gens.index].groupby(gens.carrier).sum()
        bus_gen_energy.loc[bus_name, gen_energy_at_bus.index] = gen_energy_at_bus.values

# bus_gen_energy: rows are bus names, columns are generator types, values are total generation per bus/type

bus_gen_energy.sort_values(by="load", ascending=False)

# Calculate load share of each bus, sort ascendingly
total_load = bus_gen_energy["load"].sum()
bus_gen_energy["load_share"] = bus_gen_energy["load"] / total_load
bus_gen_energy = bus_gen_energy.sort_values(by="load_share", ascending=False)

bus_gen_energy